# Notebook 3 — A2C Training with Checkpoints

Trains A2C on `ALE/Pong-v5` using 4 parallel environments.
Checkpoints are saved every 100k environment steps; evaluation runs every 50k steps.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/RL_Models'
except ImportError:
    SAVE_DIR = 'models'

import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import gymnasium as gym
import ale_py
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

gym.register_envs(ale_py)

ENV_ID          = 'ALE/Pong-v5'
RUN_NAME        = 'a2c_pong_default'
N_ENVS          = 4
TOTAL_TIMESTEPS = 2_000_000

In [ ]:
# n_envs parallel envs — each collects steps simultaneously
vec_env  = make_atari_env(ENV_ID, n_envs=N_ENVS, seed=42)
vec_env  = VecFrameStack(vec_env, n_stack=4)

eval_env = make_atari_env(ENV_ID, n_envs=1, seed=0)
eval_env = VecFrameStack(eval_env, n_stack=4)

run_dir = os.path.join(SAVE_DIR, RUN_NAME)

# save_freq is per-env steps, so divide by n_envs to hit ~100k total steps
checkpoint_cb = CheckpointCallback(
    save_freq=max(100_000 // N_ENVS, 1),
    save_path=run_dir,
    name_prefix='a2c',
)
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(run_dir, 'best'),
    log_path=os.path.join(run_dir, 'eval_logs'),
    eval_freq=max(50_000 // N_ENVS, 1),
    n_eval_episodes=20,
    deterministic=True,
)

In [ ]:
model = A2C(
    'CnnPolicy', vec_env,
    learning_rate=7e-4,
    n_steps=5,          # steps per env before each update
    gamma=0.99,
    gae_lambda=1.0,     # no GAE smoothing (pure n-step return)
    ent_coef=0.01,      # entropy bonus encourages exploration
    vf_coef=0.25,
    max_grad_norm=0.5,
    tensorboard_log=os.path.join(SAVE_DIR, 'tensorboard'),
    verbose=1,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[checkpoint_cb, eval_cb],
    tb_log_name=RUN_NAME,
)

model.save(os.path.join(run_dir, 'a2c_final'))
vec_env.close()
eval_env.close()
print('Done!')

In [ ]:
# List saved checkpoints
import glob
ckpts = sorted(glob.glob(os.path.join(run_dir, 'a2c_*.zip')))
print(f'Saved checkpoints ({len(ckpts)}):')
for c in ckpts:
    print(' ', c)

## Hyperparameter Experiment — More Parallel Envs

Re-run with `N_ENVS=8` and `ent_coef=0.001` to compare against the default run above.
Change `RUN_NAME` so the two runs don't overwrite each other.

In [ ]:
RUN_NAME_2 = 'a2c_pong_8envs_lowent'
N_ENVS_2   = 8

vec_env2  = make_atari_env(ENV_ID, n_envs=N_ENVS_2, seed=42)
vec_env2  = VecFrameStack(vec_env2, n_stack=4)
eval_env2 = make_atari_env(ENV_ID, n_envs=1, seed=0)
eval_env2 = VecFrameStack(eval_env2, n_stack=4)

run_dir2 = os.path.join(SAVE_DIR, RUN_NAME_2)

checkpoint_cb2 = CheckpointCallback(
    save_freq=max(100_000 // N_ENVS_2, 1),
    save_path=run_dir2,
    name_prefix='a2c',
)

model2 = A2C(
    'CnnPolicy', vec_env2,
    learning_rate=7e-4,
    n_steps=5,
    gamma=0.99,
    ent_coef=0.001,   # lower entropy → less random exploration
    vf_coef=0.25,
    max_grad_norm=0.5,
    tensorboard_log=os.path.join(SAVE_DIR, 'tensorboard'),
    verbose=1,
)

model2.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_cb2,
    tb_log_name=RUN_NAME_2,
)
model2.save(os.path.join(run_dir2, 'a2c_final'))
vec_env2.close()
eval_env2.close()
print('Experiment 2 done!')